In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from tqdm import tqdm

In [ ]:
# =========================
# Config
# =========================
CSV_PATH = "../data/csv/daangn_폴로.csv"
ID_COL = "id"
HREF_COL = "href"

SAVE_DIR = Path("../data/images/폴로")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://www.daangn.com"
UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0.0.0 Safari/537.36"
)

HEADERS_BASE: dict[str, str] = {
    "User-Agent": UA,
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
    "Referer": BASE,
}

HEADERS_HTML = {
    **HEADERS_BASE,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

HEADERS_IMG = {
    **HEADERS_BASE,
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

TIMEOUT = 20
RETRIES = 4

# ✅ 9000개면 무리한 병렬 금지: 6~12 추천
MAX_WORKERS = 8

# ✅ 패턴 완화 (각 아이템 처리 사이 랜덤 슬립)
SLEEP_MIN = 0.10
SLEEP_MAX = 0.40

# =========================
# Detail parser (네 코드 기반)
# =========================
REMIX_RE = re.compile(
    r"window\.__remixContext\s*=\s*(\{.*?\})\s*;\s*</script>", re.S
)

def extract_remix_context(html: str) -> dict:
    m = REMIX_RE.search(html)
    if not m:
        raise ValueError("remixContext not found")
    return json.loads(m.group(1))

def dig(obj: object, path: list[str]) -> object | None:
    cur = obj
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return None
        cur = cur[p]
    return cur

def parse_detail_image_urls(html: str) -> list[str]:
    remix = extract_remix_context(html)
    loader = dig(remix, ["state", "loaderData"])
    if not isinstance(loader, dict):
        raise ValueError("loaderData missing")

    route_key = next((k for k in loader if "buy_sell_id" in k), None)
    if route_key is None:
        raise ValueError("route key missing")

    product = dig(loader, [route_key, "product"])
    if not isinstance(product, dict):
        raise ValueError("product missing")

    images = product.get("images")
    image_urls: list[str] = []

    if isinstance(images, list):
        for img in images:
            if isinstance(img, str) and img:
                image_urls.append(img)
            elif isinstance(img, dict):
                u = img.get("url") or img.get("imageUrl") or img.get("src")
                if isinstance(u, str) and u:
                    image_urls.append(u)

    return image_urls

def fetch_image_urls(session: requests.Session, href: str, cookie: str | None = None) -> list[str]:
    url = href if href.startswith("http") else urljoin(BASE, href)
    headers = HEADERS_HTML.copy()
    if cookie:
        headers["Cookie"] = cookie

    r = session.get(url, headers=headers, timeout=TIMEOUT)
    r.raise_for_status()
    return parse_detail_image_urls(r.text)

def infer_ext(full_url: str) -> str:
    ext = Path(urlparse(full_url).path).suffix.lower()
    if ext not in [".jpg", ".jpeg", ".png", ".webp"]:
        ext = ".jpg"
    return ext

def download_main_image(
    session: requests.Session,
    image_url: str,
    item_id: str,
    cookie: str | None = None,
) -> tuple[bool, str]:
    if not image_url or not item_id:
        return False, "missing_input"

    full_url = image_url if image_url.startswith("http") else urljoin(BASE, image_url)
    ext = infer_ext(full_url)

    save_path = SAVE_DIR / f"{item_id}{ext}"
    if save_path.exists() and save_path.stat().st_size > 0:
        return True, "skip_exists"

    headers = HEADERS_IMG.copy()
    if cookie:
        headers["Cookie"] = cookie

    r = session.get(full_url, headers=headers, timeout=TIMEOUT, stream=True)
    if r.status_code != 200:
        return False, f"HTTP {r.status_code}"

    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 64):
            if chunk:
                f.write(chunk)

    if save_path.stat().st_size == 0:
        save_path.unlink(missing_ok=True)
        return False, "empty_file"

    return True, "ok"


# =========================
# Worker with retries + jitter
# =========================
def process_one(item_id: str, href: str, cookie: str | None) -> tuple[bool, str]:
    # 세션은 스레드당 1개 생성해서 쿠키/헤더 유지
    with requests.Session() as session:
        for attempt in range(RETRIES):
            try:
                # ✅ 아이템별 랜덤 슬립 (패턴 완화)
                time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

                image_urls = fetch_image_urls(session, href, cookie=cookie)
                if not image_urls:
                    return False, "no_images"

                # ✅ 메인 1장만
                return download_main_image(
                    session=session,
                    image_url=str(image_urls[0]),
                    item_id=item_id,
                    cookie=cookie,
                )

            except requests.HTTPError as e:
                # 429/403 등도 여기로 들어올 수 있음
                backoff = 1.3 * (attempt + 1) + random.uniform(0.0, 0.7)
                time.sleep(backoff)
                last = f"HTTPError: {e}"
            except Exception as e:
                backoff = 1.3 * (attempt + 1) + random.uniform(0.0, 0.7)
                time.sleep(backoff)
                last = repr(e)

        return False, f"failed_after_retries: {last}"


def main(cookie: str | None = None):
    df = pd.read_csv(CSV_PATH)
    df = df[[ID_COL, HREF_COL]].dropna()

    items = list(zip(df[ID_COL].astype(str).tolist(), df[HREF_COL].astype(str).tolist()))

    ok = skip = fail = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_one, item_id, href, cookie) for item_id, href in items]

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Download main images"):
            success, msg = fut.result()
            if success and msg == "skip_exists":
                skip += 1
            elif success:
                ok += 1
            else:
                fail += 1

    print(f"\nDone. ok={ok}, skip={skip}, fail={fail}")
    print(f"Saved to: {SAVE_DIR.resolve()}")

cookie = r'_gcl_au=1.1.1444086516.1772091281; _ga=GA1.1.1864578218.1772091281; _fbp=fb.1.1772091280970.177480178187518519; _clck=1v9lldb%5E2%5Eg43%5E0%5E2248; search_region=eyJyZWdpb25JZCI6IjYwMzQiLCJyZWdpb25OYW1lIjoi7IK87ISx64%2BZIiwiY291bnRyeUNvZGUiOiJrciJ9; ph_phc_FqtkYFQ4JIDKz1sAraoJA02LALWTkJh8F8okhjnxd3Z_posthog=%7B%22distinct_id%22%3A%22019c98df-0c6b-78ed-9837-b90a124b3fa1%22%2C%22%24sesid%22%3A%5B1772677710535%2C%22019cbbd3-0526-7f55-b1b1-27519dc26a02%22%2C1772677694758%5D%2C%22%24initial_person_info%22%3A%7B%22r%22%3A%22https%3A%2F%2Fwww.google.com%2F%22%2C%22u%22%3A%22https%3A%2F%2Fwww.daangn.com%2Fkr%2F%22%7D%7D; _clsk=h9gh3v%5E1772677711055%5E2%5E0%5Es.clarity.ms%2Fcollect; _ga_KNCHQ0QW6Q=GS2.1.s1772677694$o12$g1$t1772677711$j43$l0$h0'
main(cookie=cookie)

Download main images:  36%|███▌      | 3244/8970 [21:52<3:54:39,  2.46s/it]